In [ ]:
import pandas as pd 
import numpy as np 

In [ ]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [ ]:
movies.head(1)

In [ ]:
movies = movies.merge(credits,on='title')

In [ ]:
#genres 
#movie_id
#keywords
#title
#overview
#cast
#crew

movies = movies[['movie_id','title','keywords','overview','genres','cast','crew']]

In [ ]:
movies.info()

In [ ]:
movies.isnull().sum()

In [ ]:
#overview has 3 null so just drop them
movies.dropna(inplace=True)

In [ ]:
movies.duplicated().sum()

In [ ]:
movies.iloc[0].genres

In [ ]:
#'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

#convert this messy into basic
#{action,adventure,fantasy,science fic}

#but therre is a problem that our genres column is a 'string list' which is not supported so convert it into "list" usingg
# using import ast
# ast.literal_eval

In [ ]:
import ast
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [ ]:
movies['genres'].apply(convert)

In [ ]:
# save the new genre column
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
#use the same loop for keywords
movies['keywords'].apply(convert)

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
#see sorted genres and keywords
movies.head()

In [ ]:
#now we extract main 3 actors name from cast 
# using same loop with a itration to stop at 3rd actor name
import ast
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
          L.append(i['name'])
          counter+=1
        else:
           break
    return L

In [ ]:
movies['cast'].apply(convert3)

In [ ]:
movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
movies.head()

In [ ]:
#now we need only directors name
import ast 
def fetch_director(obj):
    L = []
    
    for i in ast.literal_eval(obj):
        if i['job']== 'Director': 
          L.append(i['name'])
          break
    return L

In [ ]:
movies['crew'].apply(fetch_director)

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
#movies is overview is also a string that we need to convert into a list
movies['overview'].apply(lambda x:x.split())

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies.head()

In [ ]:
# We know the space between names, chings name etc couse misunderstanding for model
# Like div pareta name, it considered as 2 tags (div) (pareta) but we may also have aysuh pareta in data so ...
# (ayush)(pareta) again 2 tags but notice that now data have 2 similar tags { pareta } which couse confusion for model
# The only solution for this problem is to remove : "space"

movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","") for i in x]) 
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","") for i in x]) 
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ","") for i in x])


In [ ]:
movies.head()

In [ ]:
# Mix all the columns to create one only named : Tags
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies.head()

In [ ]:
# Now we dont need this data frame so we made a new dataframe
new_df = movies[['movie_id','title','tags']]

In [ ]:
new_df

In [ ]:
# Now we are done with cleaning ,so all lists need to converted into string
new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df.head()

In [ ]:
# Recommended by pro's to convert all into lower case 
new_df['tags'].apply(lambda x:x.lower())

new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

In [ ]:
# Done with sorting , cleaning , managing etc

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer                                 
cv = CountVectorizer(max_features=5000,stop_words='english')

In [ ]:
cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vectors[0]

In [ ]:
cv.get_feature_names_out()

In [ ]:
# our data have many similar words like - dance ,dancer, dancing - love,loved,lover but for our this project the route word 
# Love or Dance is main so ,, stem libriry is used

In [ ]:
!pip install nltk 

In [ ]:
import nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()


In [ ]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)    

In [ ]:
ps.stem('dancing')

In [ ]:
ps.stem('dancing')

In [ ]:
new_df['tags'].apply(stem)

In [ ]:
new_df['tags'] = new_df['tags'].apply(stem)

In [ ]:
# Use this function to check distance/ similarity in movies
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_similarity(vectors)

In [ ]:
cosine_similarity(vectors).shape

In [ ]:
# save all into a matric
similarity = cosine_similarity(vectors)

In [ ]:
similarity

In [ ]:
# Our data is ready for recommender, so starting here we are code for it.
# we need to hold all the index values after sorting ,so use [enumerate]
list(enumerate(similarity[0]))

In [ ]:
# Lamba function is used so sorting starts from movie distance not index values
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])

In [78]:
# Now we need to create a function, we provide 1 movie to it then it gives us 5 similar movie names
def recommend(movie):
    
    # Find the movie by converting both dataset titles and user input to lowercase
    # This prevents errors if someone types 'batman begins' instead of 'Batman Begins'
    match = new_df[new_df['title'].str.lower() == movie.lower()]

    # Check if the movie actually exists in our dataset
    # If the match dataframe is empty, it means the movie is not in our data
    if match.empty:
        print("Movie not found in the database. Please check the spelling.")
        return # Exit the function here to prevent the IndexError from happening
        
    # Get the index number of the matched movie from our dataset
    movie_index = match.index[0]
    
    # Fetch the array of similarity scores for this specific movie
    distances = similarity[movie_index]
    
    # Enumerate keeps track of original movie indexes while we sort
    # The lambda function tells it to sort by the similarity score (x[1]) instead of index (x[0])
    # reverse=True sorts in descending order (highest similarity at the top)
    # [1:6] skips the first movie (which is a 100% match with itself) and gets the next 5 recommendations
    movie_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x:x[1])[1:6]
    
    # Loop through the top 5 similar movies
    for i in movie_list:
        # i[0] gives the movie index, which we use to print the actual title from the dataframe
        print(new_df.iloc[i[0]].title)

In [79]:
# All set 
recommend('Avatar')

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem


In [85]:
# But we need name not index so we put this into our upper funtion
# [new_df.iloc[539].title]
recommend('batman begins')

The Dark Knight
The Dark Knight Rises
Batman
Batman
Batman & Robin


In [86]:
recommend('Titan')

Movie not found in the database. Please check the spelling.


In [87]:
import pickle

# dataframe (new_df) save here
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))

# Now similarity matrix save here
pickle.dump(similarity, open('similarity.pkl', 'wb'))